# ⚽ EZStats — Living Product Plan

> **Single source of truth.** All plans, updates, and status changes go here only.

## 📋 Session Rules

| # | Rule |
|---|------|
| 1 | **Read this file (`docs/product_plan.ipynb`) first** every Claude session before doing anything |
| 2 | **Finish one step fully** (test it, verify it) before starting the next |
| 3 | **Stuck on a step?** Diagnose and fix it — never skip ahead |
| 4 | **End of session** — update checkboxes and statuses here |

## 🎯 Where We Are Right Now

```
Phase 0 █████████░  90%
  ✅ 0-A  Run scripts fixed + viewer tested — bugs found, passes wrong (tracking)
  ✅ 0-C  match_report.json + viewer.html built
  ⏸ 0-B  event_spotter.py rewrite  ← waiting for PCBAS-2026 training
           ⚠️  Training collapsed — model predicts DRIVE 100%
           Fix: add class_weights to CrossEntropyLoss in training notebook

Phase 1 ░░░░░░░░░░   0%  ← REVERTED TO GOOD BASELINE
  🔄 1-A  Tracking — siglip_merge.py written but NOT wired in (cli.py reverted)
           Good baseline: outputs/20260508_211913
           ID swap at overlap still unresolved — siglip merge is the plan

NEXT ACTION: Wire merge-tracklets back into cli.py cleanly, test on 20260508 run dir
```

### Known Good Run
`outputs/20260508_211913` — best tracking + team classification so far.  
Only known issue: player ID swap during overlap (frames ~297–330, track 13).

### 🐛 Bugs Found in Viewer (2026-05-12)

| Bug | Root Cause | Status |
|-----|-----------|--------|
| Pass #10→#1 wrong (correct: #10→#2→#209) | #209 = keeper re-born after overlap | 🔧 1-A |
| #20→#1, #5→#24 wrong passes | IDs swap when players cross | 🔧 1-A |
| Keeper long ball classified as pass | Ball flies far, no receiver | 📋 1-B |
| High-numbered IDs (209) mid-video | Tracker spawns new track on occlusion | 🔧 1-A |

### ⚠️ PCBAS-2026 Training Collapse
Model predicts DRIVE for 100% of samples. Root cause: class imbalance, no weighted loss.  
Fix in `train_pcbas2026_player.ipynb`:
```python
class_weights = class_counts.sum() / (len(class_counts) * class_counts)
criterion = nn.CrossEntropyLoss(weight=class_weights.to(device))
```
Retrain from scratch with fix applied before downloading model.pt.

### ✅ Tracker Testing Summary

| Config | Result | Reason |
|--------|--------|--------|
| `bytetrack_football.yaml` (original) | ❌ ID swap | `match_thresh: 0.85` too strict |
| `bytetrack_football_v2.yaml` | ❌ Still swaps | IoU-only, no camera motion compensation |
| `botsort_football.yaml` (with ReID) | ❌ Still swaps | OSNet trained on pedestrians |
| **`botsort_football.yaml` (ReID off)** | **✅ Chosen base** | **GMC handles camera pan** |

## 📊 Current Status vs Footovision

| Feature | Footovision | EZStats | Status |
|:--------|:-----------:|:-------:|:------:|
| Player + ball + referee detection | ✅ | works but misses frames, new videos | ⚠️ |
| Homography / top-down pitch | ✅ | not 100% — bad keypoints warp pitch | ⚠️ |
| **Player ID tracking — occlusion** | ✅ stable | **IDs swap when players overlap** | ❌ |
| Team classification | ✅ | breaks on ID re-assign, some frames wrong | ⚠️ |
| Voronoi territorial control | ✅ | wrong — depends on team classification | ⚠️ |
| Basic possession stats | ✅ | wrong — depends on tracking bugs above | ⚠️ |
| Three-phase event detection | ✅ | rewritten, passes wrong due to tracking | ⚠️ |
| Clearance / keeper long ball detection | ✅ | misclassified as pass | ❌ |
| 9-class Transformer (PASS/SHOT/CROSS…) | ✅ | training in Colab | 🔄 |
| Player heatmaps | ✅ | not built | ❌ |
| Per-player speed & distance (meters) | ✅ | pixel hack only — wrong numbers | ❌ |
| Event highlight clips | ✅ | not built | ❌ |
| Pass network | ✅ | not built | ❌ |
| Debug viewer / match_report.json | — | built ✅ | ✅ |
| Cloud API | ✅ | offline only | ❌ |
| Formation detection | ✅ | not built | ❌ |
| xG model | ✅ | not built | ❌ |
| Real-time processing | ✅ | ~20 min / 30s on CPU | ❌ |
| Mobile dashboard | ✅ | not built | ❌ |
| **Affordable SE Asia pricing** | ❌ enterprise | **our core edge** | ✅ |

---
## 🚀 Phase 0 — Fix Before Any Run

### 0-A · Test Rule-Based Events

- [x] Remove broken `--event-model-name` from `run_full_pipeline.ps1` + `run_new_video.ps1`
- [x] Run `.\.run_full_pipeline.ps1` → opened viewer, loaded `match_report.json`
- [ ] **Passes still wrong** → blocked by tracking ID swap bug (fix in Phase 1-A first)
- [ ] Re-run after tracking fix → manually count passes → target within 25% of truth

### 0-B · Rewrite event_spotter.py for PCBAS-2026 ⏸ WAITING FOR TRAINING

> Training notebook: `docs/train_pcbas2026_player.ipynb`  
> Download when done: Colab Drive → `artifacts/training/event_spotter_pcbas2026/model.pt`

**What's wrong with the current code vs what it should be:**

| | ❌ Current | ✅ Should Be |
|---|---|---|
| Architecture | Bi-LSTM | Transformer Encoder (4-layer, 8-head) |
| Classes | 14 BAS-2025 | **9 PCBAS-2026** (different order!) |
| Window | 15 frames | **25 frames** |
| Inference | Global frame features | **Per-player crop features** |
| Output head | Single (action) | **Dual (action + team)** |
| Save path | `event_spotter_bas2025` | **`event_spotter_pcbas2026`** |

**Correct class list (order matters for loading weights):**
```python
BALL_ACTION_CLASSES = [
    'background',  # 0
    'DRIVE',       # 1
    'PASS',        # 2
    'CROSS',       # 3
    'SHOT',        # 4
    'HEADER',      # 5
    'THROW IN',    # 6
    'TACKLE',      # 7
    'BLOCK',       # 8
]
```

**Correct architecture:**
```python
self.proj        = nn.Linear(512, 512)
self.pe          = PositionalEncoding(512, dropout=0.1)
self.encoder     = nn.TransformerEncoder(
    nn.TransformerEncoderLayer(512, nhead=8, dim_feedforward=2048,
                               dropout=0.1, batch_first=True, norm_first=True), 4
)
self.shared      = nn.Sequential(nn.Linear(512, 256), nn.ReLU(), nn.Dropout(0.1))
self.action_head = nn.Sequential(nn.Linear(256, 64), nn.ReLU(), nn.Dropout(0.2), nn.Linear(64, 9))
self.team_head   = nn.Linear(256, 3)   # 0=background, 1=left-team, 2=right-team
```

**Checklist (do after model.pt is downloaded):**
- [ ] Rewrite `_EventSpotterModel` to Transformer above
- [ ] Update `BALL_ACTION_CLASSES` → 9 PCBAS-2026 classes
- [ ] Update `WINDOW = 25`
- [ ] Update `_CLASS_TO_EVENT` + `_CLASS_THRESHOLDS` for 9 new classes
- [ ] Rewrite inference to be player-centric (per-player crop features, not frame features)
- [ ] Update `fuse_events()` class names
- [ ] Uncomment `--event-model-name` in `run_full_pipeline.ps1` + `run_new_video.ps1`

### 0-C · Debug Dashboard

- [x] `outputs/<timestamp>/match_report.json` written after every pipeline run
- [x] `outputs/viewer.html` — open in browser, load JSON, see all stats visually
- [ ] Re-run after tracking fix → open viewer → confirm passes match video

---
## 🔧 Phase 1 — Make Existing Features Actually Work
*Target: Week 1–2*

### 🔁 1-A · Tracking Stability — SigLIP Tracklet Merger  ← NEXT

> **Decision:** Use BoT-SORT with `with_reid: false` as base tracker.  
> GMC handles camera pan. SigLIP merger handles post-hoc ID fixing.

**Pipeline order after merge-tracklets is wired in:**
```
analyze
  → prepare-appearance        (generates SigLIP crop embeddings)
  → cluster-teams
  → apply-team-clusters
  → merge-tracklets           ← NEW: remap split IDs using SigLIP similarity
  → render-stats-video
  → render-spatial-video
```

**How merge-tracklets works:**
- Reads `tracks_with_teams.json` + SigLIP embeddings from `player_crops/`
- For each pair of tracklets: (a) no time overlap, (b) same team, (c) cosine similarity > 0.82 → remap higher ID → lower ID
- Writes `tracks_merged.json` + rewrites `tracks_with_teams.json`

**Current state:**
- `src/ez_worker/postprocess/siglip_merge.py` — written, numpy `or` bug fixed, NOT wired in
- `cli.py` — reverted (no merge-tracklets command)
- Both run scripts — no merge-tracklets step

**Checklist:**
- [x] ByteTrack v2 tested → ❌ still swaps
- [x] BoT-SORT with ReID tested → ❌ OSNet can't distinguish football players
- [x] Decided: BoT-SORT (ReID off, GMC on) — configs + run scripts updated
- [x] `siglip_merge.py` written + numpy bug fixed (use explicit `is None` checks)
- [ ] Add `merge-tracklets` command back to `cli.py`
- [ ] `render-stats-video` reads `tracks_merged.json` if present (fallback to `tracks_with_teams.json`)
- [ ] Add step to `run_new_video.ps1` + `run_full_pipeline.ps1` after `apply-team-clusters`
- [ ] Run on `outputs/20260508_211913` run dir → open `stats_video.mp4` → confirm IDs stable
- [ ] Open viewer → confirm max ID ≤ 30, passes make sense

### 🎯 1-B · Event Detection — Fix False Passes + High Ball  ✅ DONE 2026-05-12

**Changes made:**
- `possession_distance_threshold_px` 65 → 50 (tighter possession zone)
- `pass_min_speed_px_per_s` 150 → 200 (higher speed needed to enter IN_FLIGHT)
- `ball_direction_change_min_deg` 20 → 35 (stronger direction change for slow-possession pass)
- **Arc detection** (`_is_high_arc`): fits parabolic trajectory to ball y-positions during IN_FLIGHT
  - Peak (min y in image) must occur in middle 15–85% of flight = genuine arc
  - Arc height > 4% of frame height = real high ball
- **`clearance` event**: flight > 1.2s AND arc detected AND no receiver → replaces `shot_attempt` for GK kicks
- **`long_ball` event**: flight > 1.2s AND arc detected AND receiver found → replaces `pass` for high balls
- **Manual track merge**: place `manual_track_merges.json` in run dir, e.g. `[[209, 10]]`
  - Remaps track 209 → 10 in tracks, stats, and events.json
  - Combines numeric stats (touches, passes, shots, distance) for merged tracks
- **match_report.json** now rebuilt after `apply-team-clusters` → fixes team showing as "?"
  - Includes new `total_long_balls` and `total_clearances` in summary

**Checklist:**
- [x] Tighten possession distance (65 → 50px)
- [x] Raise pass speed threshold (150 → 200 px/s)
- [x] Raise direction change threshold (20 → 35°)
- [x] `clearance` event type implemented
- [x] `long_ball` event type implemented
- [x] Physics arc detection using ball y-trajectory
- [x] Manual track merge via `manual_track_merges.json`
- [x] match_report.json rebuilt after apply-team-clusters (fixes "?" team)
- [ ] Run on `19PassesAndMasonGoal.mp4` → manually count passes → target within 25% of truth
- [ ] Verify clearance fires for keeper long ball, not shot_attempt
- [ ] Verify no false passes from loose ball rolling near multiple players
- [ ] Add `clearance`/`long_ball` badge styles to `viewer.html`

### 🔍 1-C · Detection Quality Gate
> Current detector is YOLOv8n trained on only 298 images of Bundesliga footage.  
> Everything else depends on detection being reliable.

- [ ] Players detected consistently — no player vanishing for >10 consecutive frames
- [ ] Ball detected in ≥70% of frames
- [ ] All 22 players on pitch detected at same time

> **If gate fails → retrain detector first:**
> - Annotate frames from failing video in Roboflow format
> - Upgrade `yolov8n` → `yolov8s` for better accuracy
> - Target: 1000+ images, 4 classes (player / goalkeeper / referee / ball)
> - `yolo detect train model=yolov8s.pt data=data/datasets/roboflow/detector/data.yaml epochs=100`

### 🗺️ 1-D · Homography — Make It Stable

- [ ] Fix H averaging in `stats_video.py` — compute `np.mean(list(H_deque), axis=0)` (currently just uses last H)
- [ ] Add H quality check — `abs(np.linalg.det(H)) > 0.1` before accepting
- [ ] Clip pitch projections to `[0, 10500] × [0, 6800]` cm after `perspectiveTransform`
- [ ] Plumb pitch coordinates into `detect_events()` via `pipeline.py` (unlocks cm-space event detection)
- [ ] Visual check: Voronoi on `08fd33_4.mp4` looks geometrically correct

### 👕 1-E · Team Classification — Reduce Misassignment

- [ ] Fix GK exclusion — filter by pitch position (closest to goal post) not just detection label
- [ ] Add UMAP guard — if `n_samples < 20`, fall back to cosine similarity clustering
- [ ] Add temporal team lock — once a `track_id` is assigned a team, never flip it mid-track
- [ ] Visual check: team colors stable for ≥90% of frames on both test videos

### 📈 1-F · Possession Stats — Real Numbers

- [ ] Replace pixel distance in `stats.py` with pitch-coordinate distance in meters
- [ ] Validate: possession % on `19PassesAndMasonGoal.mp4` should not be 50-50

---
## ✨ Phase 2 — First Sellable Features
*Target: Week 3–4*

### 🌡️ Player Heatmaps
- [ ] Accumulate `(x_cm, y_cm)` per `tracker_id` in render loop
- [ ] New: `src/ez_worker/analytics/heatmap.py` — Gaussian KDE on 105×68 pitch grid
- [ ] Output PNG per player + team aggregate
- [ ] Add heatmap paths to `match_report.json` + show in `viewer.html`

### 🏃 Per-Player Speed & Distance
- [ ] Distance = sum of Euclidean distance in cm between frames → meters
- [ ] Max speed = `max(Δcm / Δseconds)`, capped at 1200 cm/s (43 km/h)
- [ ] Add to `TrackStats` and `match_report.json`

### 🎬 Event Highlight Clips
- [ ] New: `src/ez_worker/io/clip_exporter.py`
- [ ] On SHOT / GOAL / CORNER / FREE KICK — cut `[event_frame - 2s : event_frame + 3s]`
- [ ] Named: `GOAL_45m23s.mp4`, `SHOT_32m11s.mp4`
- [ ] Add clip paths to events in `match_report.json`

### 🕸️ Pass Network
- [ ] New: `src/ez_worker/analytics/pass_network.py`
- [ ] Directed graph: `player_A → player_B` count from PASS events
- [ ] Output JSON edge list + static pitch diagram image
- [ ] Show in `viewer.html`

---
## 📱 Phase 3 — Friend's Frontend Ready
*Target: Week 5–6*

- [ ] Hand `match_report.json` schema to frontend friend
- [ ] Test on `BrightonGoal.mp4` — first untested video — no crashes, goal detected
- [ ] Heatmaps + clips + pass network all working end-to-end

---
## ☁️ Phase 4 — Cloud API
*Target: Week 7–8*

- [ ] New: `src/ez_worker/api/main.py` — FastAPI
- [ ] `POST /analyze` → upload video → `job_id`
- [ ] `GET /job/{id}` → status + download links
- [ ] Deploy on RunPod / Lambda Labs GPU — 90-min match in < 30 min
- [ ] Simple web UI: drag-and-drop → progress bar → download results

---
## 📐 Phase 5 — Analytics Gap (Month 2)

- [ ] **xG model** — shot `(x_cm, y_cm)` + event class → logistic regression on SoccerNet data
- [ ] **Formation detection** — cluster positions at kickoff/set pieces → 4-4-2 / 4-3-3 etc.
- [ ] **Pressing intensity** — count players within 1000 cm of ball carrier per frame

---
## 🌏 Phase 6 — SE Asia Market Lock-In (Month 3+)

- [ ] Collect 10+ hours Myanmar National League / Thai League footage from clubs
- [ ] Fine-tune YOLO detector on local footage (different jerseys, pitch, lighting)
- [ ] Mobile-responsive dashboard (React Native or PWA)
- [ ] Pricing: **$15 / match** or **$80 / month** per club
- [ ] Onboard first 5 pilot clubs — free trial in exchange for video footage
- [ ] Localization: Burmese + Thai UI

---
## 🗂️ Key Files Reference

| File | Status | Pending Work |
|:-----|:------:|:-------------|
| `configs/botsort_football.yaml` | ✅ | Chosen base tracker — ReID off, GMC on |
| `configs/bytetrack_football_v2.yaml` | ✅ | Tested, rejected — kept for reference |
| `src/ez_worker/postprocess/tracks.py` | 🔧 | SigLIP tracklet merger (Phase 1-A) |
| `src/ez_worker/cli.py` | 🔧 | Add `merge-tracklets` command (Phase 1-A) |
| `src/ez_worker/io/stats_video.py` | 📋 | Read `tracks_merged.json`; accumulate pitch coords; fix H averaging |
| `src/ez_worker/analytics/events.py` | 📋 | Add `clearance` + `long_ball` event types (Phase 1-B) |
| `src/ez_worker/analytics/event_spotter.py` | ⏸ | Full rewrite → PCBAS-2026 Transformer (Phase 0-B, after training) |
| `src/ez_worker/analytics/stats.py` | 📋 | Distance in real meters (Phase 1-F) |
| `src/ez_worker/spatial/view_transformer.py` | 📋 | H quality check + projection bounds clamp (Phase 1-D) |
| `src/ez_worker/appearance/team_assignment.py` | 📋 | GK exclusion fix; UMAP guard (Phase 1-E) |
| `src/ez_worker/outputs/writer.py` | ✅ | match_report.json done |
| `run_full_pipeline.ps1` + `run_new_video.ps1` | ✅ | BoT-SORT active; add `merge-tracklets` step after impl |
| `outputs/viewer.html` | 📋 | Add `clearance`/`long_ball` badge styles (Phase 1-B) |
| `src/ez_worker/analytics/heatmap.py` | 📋 | New — Phase 2 |
| `src/ez_worker/io/clip_exporter.py` | 📋 | New — Phase 2 |
| `src/ez_worker/analytics/pass_network.py` | 📋 | New — Phase 2 |
| `src/ez_worker/api/main.py` | 📋 | New — Phase 4 |